In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import random
from collections import deque
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

print("=== DAY 33 — FULL DQN TRAINING LOOP (Fixed & Clean) ===\n")

# =============================================
# 1. REWARD FUNCTION
# =============================================
class RewardFunction:
    def __init__(self, cost=0.0005, risk_penalty=0.08, drawdown_penalty=0.25):
        self.cost = cost
        self.risk_penalty = risk_penalty
        self.drawdown_penalty = drawdown_penalty
    
    def compute(self, current_ret, prev_position, new_position, drawdown):
        strategy_ret = prev_position * current_ret
        cost = abs(new_position - prev_position) * self.cost
        risk_penalty = self.risk_penalty * (strategy_ret ** 2)
        dd_penalty = self.drawdown_penalty * abs(drawdown) if drawdown < -0.05 else 0
        reward = strategy_ret - cost - risk_penalty - dd_penalty
        net_ret = strategy_ret - cost
        return reward, net_ret

# =============================================
# 2. ENVIRONMENT
# =============================================
class QuantumAlphaEnv:
    def __init__(self, data, initial_balance=1000000):
        self.data = data.reset_index(drop=True)
        self.max_steps = len(self.data) - 1
        self.initial_balance = initial_balance
        self.reward_fn = RewardFunction()
        
        self.feature_cols = [
            "mom_20_norm", "vol_signal_norm", "trend_signal_norm",
            "dd_signal_norm", "vix_signal_norm", "breadth_signal_norm"
        ]
        
        self.reset()
    
    def reset(self):
        self.current_step = 0
        self.position = 0
        self.balance = self.initial_balance
        self.peak_balance = self.initial_balance
        return self._get_observation()
    
    def _get_observation(self):
        row = self.data.iloc[self.current_step]
        return row[self.feature_cols].values.astype(np.float32)
    
    def step(self, action):
        new_position = {0: 0, 1: 1, 2: -1}[action]
        prev_position = self.position
        current_ret = self.data.iloc[self.current_step]["nifty_ret"]
        current_drawdown = (self.balance / self.peak_balance) - 1 if self.peak_balance > 0 else 0
        
        reward, net_ret = self.reward_fn.compute(
            current_ret, prev_position, new_position, current_drawdown
        )
        
        self.balance *= (1 + net_ret)
        self.peak_balance = max(self.peak_balance, self.balance)
        self.position = new_position
        self.current_step += 1
        done = self.current_step >= self.max_steps
        
        return self._get_observation(), reward, done, {
            "balance": self.balance,
            "position": self.position
        }

# =============================================
# 3. LOAD DATA + CREATE ENVIRONMENT
# =============================================
state_data = pd.read_csv("../data/state_dataset.csv", index_col="Date", parse_dates=True)
market_data = pd.read_csv("../data/market_master.csv", index_col="Date", parse_dates=True)
data = state_data.join(market_data[["nifty_ret"]], how="inner")

env = QuantumAlphaEnv(data)
print("✅ Environment created")

# =============================================
# 4. DQN MODEL
# =============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )
    
    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy_net = DQN(env.observation_space.shape[0] if hasattr(env, 'observation_space') else len(env.feature_cols), 3).to(device)
target_net = DQN(env.observation_space.shape[0] if hasattr(env, 'observation_space') else len(env.feature_cols), 3).to(device)
target_net.load_state_dict(policy_net.state_dict())
target_net.eval()

print("✅ DQN networks initialized")

# =============================================
# 5. REPLAY BUFFER (Fixed with __len__)
# =============================================
class ReplayBuffer:
    def __init__(self, capacity=100000):
        self.buffer = deque(maxlen=capacity)
    
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        states, actions, rewards, next_states, dones = zip(*batch)
        return (
            torch.FloatTensor(np.array(states)).to(device),
            torch.LongTensor(np.array(actions)).to(device),
            torch.FloatTensor(np.array(rewards)).to(device),
            torch.FloatTensor(np.array(next_states)).to(device),
            torch.FloatTensor(np.array(dones)).to(device)
        )
    
    def __len__(self):
        return len(self.buffer)

memory = ReplayBuffer()
print("✅ Replay Buffer ready")

# =============================================
# 6. TRAINING SETUP
# =============================================
optimizer = optim.Adam(policy_net.parameters(), lr=1e-4)
criterion = nn.MSELoss()

gamma = 0.99
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995
batch_size = 128
target_update = 500
num_episodes = 300

# =============================================
# 7. TRAINING LOOP (Fixed)
# =============================================
episode_rewards = []

for episode in range(num_episodes):
    state = env.reset()
    total_reward = 0
    done = False
    
    while not done:
        if random.random() < epsilon:
            action = random.randint(0, 2)
        else:
            with torch.no_grad():
                s = torch.FloatTensor(state).unsqueeze(0).to(device)
                action = policy_net(s).argmax().item()
        
        next_state, reward, done, _ = env.step(action)
        
        memory.push(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        
        # Train
        if len(memory) > batch_size:
            states, actions, rewards, next_states, dones = memory.sample(batch_size)
            
            current_q = policy_net(states).gather(1, actions.unsqueeze(1)).squeeze(1)
            
            with torch.no_grad():
                next_q = target_net(next_states).max(1)[0]
                target_q = rewards + gamma * next_q * (1 - dones)
            
            loss = criterion(current_q, target_q)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(policy_net.parameters(), 1.0)
            optimizer.step()
        
        if episode % 100 == 0 and done:
            target_net.load_state_dict(policy_net.state_dict())
    
    epsilon = max(epsilon_min, epsilon * epsilon_decay)
    episode_rewards.append(total_reward)
    
    if episode % 50 == 0:
        print(f"Episode {episode:3d} | Reward: {total_reward:8.2f} | Epsilon: {epsilon:.3f}")

# =============================================
# 8. SAVE + PLOT
# =============================================
torch.save(policy_net.state_dict(), "../models/dqn_policy_final.pth")

plt.figure(figsize=(12,5))
plt.plot(episode_rewards)
plt.title("Training - Episode Rewards")
plt.xlabel("Episode")
plt.ylabel("Total Reward")
plt.grid(True)
plt.show()

print("\n✅ Day 33 Training Loop Completed!")

=== DAY 33 — FULL DQN TRAINING LOOP (Fixed & Clean) ===

✅ Environment created
✅ DQN networks initialized
✅ Replay Buffer ready
Episode   0 | Reward:  -119.38 | Epsilon: 0.995
